# Problem 2 — student implementation interfaces and verification
**Incomplete: no custom Transformer implementation or training experiments exist.** AI prepared interfaces, tests and explanatory guidance only; disclose this assistance. All graded implementation TODOs belong to the student. `PENDING` is not success. No framework attention or Transformer layers may be called by the custom components; references below are exclusively test oracles.

The student also writes the experiment code, as confirmed on 2026-11-08. Data preparation/training/generation are not silently supplied here. The official instructor three-token example is absent; the illustrative fixture below is a diagnostic, not a claimed substitute.

## Attention contract and mathematics
`scaled_dot_product_attention(q, k, v, allowed=None)` returns `(output, weights)`.
- q/k: `(batch, heads, tokens, head_dim)`; v has the same shape for these tests.
- output: same shape as q; weights: `(batch, heads, tokens, tokens)`.
- Optional boolean `allowed`: broadcastable to weights; **True means allowed**, matching functional SDPA (opposite to the boolean mask convention of nn.MultiheadAttention).
- Scale query-key scores by the square root of head width, mask forbidden scores before row-wise softmax, then form weighted values. Use tensor operations, not library attention.
- Every query in these tests has at least one allowed key. Fully masked rows are outside this contract. No attention-weight dropout is required by this interface; block dropout is separate.


In [2]:
import math
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(101)
P2_RESULTS = {}

def check_close(actual, expected, label):
    assert isinstance(actual, torch.Tensor), label
    assert actual.shape == expected.shape, f'{label}: wrong shape'
    assert actual.dtype == expected.dtype and actual.device == expected.device, label
    assert torch.isfinite(actual).all(), f'{label}: nonfinite'
    error = (actual - expected).abs().max().item()
    assert error < 1e-5, f'{label}: max absolute error {error} >= 1e-5'
    return error

In [3]:
assert check_close(torch.ones(2), torch.ones(2), 'equal') == 0
for bad in (torch.zeros(2), torch.ones(3), torch.full((2,), float('nan'))):
    try:
        check_close(bad, torch.ones(2), 'deliberate invalid output')
    except AssertionError:
        pass
    else:
        raise AssertionError('Comparator accepted invalid output')
print('PASS: comparison helper negative controls; no student code verified')

PASS: comparison helper negative controls; no student code verified


In [4]:
def pending_or_check(name, check):
    P2_RESULTS[name] = 'In Progress'
    try:
        result = check()
    except NotImplementedError as exc:
        P2_RESULTS[name] = 'Blocked'
        print(f'PENDING {name}: {exc}')
    else:
        P2_RESULTS[name] = 'Verified'
        print(f'PASS {name}: {result}')

In [5]:
def _todo_probe():
    raise NotImplementedError('test-only unfinished probe')
pending_or_check('_probe', _todo_probe)
assert P2_RESULTS.pop('_probe') == 'Blocked'
print('PASS: unfinished checks cannot report success')

PENDING _probe: test-only unfinished probe
PASS: unfinished checks cannot report success


In [6]:
def verify_attention(candidate):
    worst = 0.0
    with torch.random.fork_rng(devices=[]):
        for dtype in (torch.float32, torch.float64):
            for seed in (101, 202):
                torch.manual_seed(seed)
                for batch, heads, length, width in ((1, 1, 3, 2), (2, 4, 7, 8)):
                    q, k, v = [torch.randn(batch, heads, length, width, dtype=dtype) for _ in range(3)]
                    causal = torch.ones(length, length, dtype=torch.bool).tril()
                    for allowed in (None, causal):
                        out, weights = candidate(q, k, v, allowed)
                        assert weights.shape == (batch, heads, length, length)
                        assert torch.isfinite(weights).all() and (weights >= 0).all()
                        worst = max(worst, check_close(weights.sum(-1), torch.ones_like(weights.sum(-1)), 'row sums'))
                        if allowed is not None:
                            assert torch.count_nonzero(weights[..., ~allowed]) == 0, 'Future information leaked'
                        reference = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, dropout_p=0.0)
                        worst = max(worst, check_close(out, reference, 'SDPA output'))
                        worst = max(worst, check_close(out, weights @ v, 'reported attention weights'))
    return {'max_abs_diff': worst}

In [7]:
# Deliberate bad output must be rejected by the actual attention verifier.
def _bad_attention(q, k, v, allowed=None):
    return torch.zeros_like(q), torch.zeros((*q.shape[:-1], k.shape[-2]), dtype=q.dtype)
try:
    verify_attention(_bad_attention)
except AssertionError:
    print('PASS: attention verifier rejects zero-sum weights')
else:
    raise AssertionError('Invalid attention accepted')

PASS: attention verifier rejects zero-sum weights


In [8]:
def scaled_dot_product_attention(q, k, v, allowed=None):
    # q/k/v: (B,H,T,Dh); allowed: bool True=visible, broadcastable to (B,H,T,T).
    # Return output (B,H,T,Dh), weights (B,H,T,T).
    # TODO (student): implement scaling, masking, row softmax and weighted values.
    raise NotImplementedError('Student attention implementation required')

In [9]:
pending_or_check('attention', lambda: verify_attention(scaled_dot_product_attention))

PENDING attention: Student attention implementation required


### Three-token illustrative hand check
Take all-zero queries and keys of width 2, and value rows `(1,0)`, `(0,2)`, `(3,4)`. Every unmasked score is zero. Without masking, all weights are 1/3 and each output row is `(4/3,2)`. With causal masking the weight rows are `(1,0,0)`, `(1/2,1/2,0)`, `(1/3,1/3,1/3)`; output rows are `(1,0)`, `(1/2,1)`, `(4/3,2)`. This checks masks and averaging; random tests above also check scaling and non-uniform attention. Replace/supplement with the instructor's example when available.

In [10]:
def verify_hand_example(candidate):
    q = torch.zeros(1, 1, 3, 2, dtype=torch.float64)
    values = torch.tensor([[[[1., 0.], [0., 2.], [3., 4.]]]], dtype=torch.float64)
    for allowed, expected_weights, expected_out in (
        (None, [[1/3]*3]*3, [[4/3, 2.]]*3),
        (torch.ones(3, 3, dtype=torch.bool).tril(),
         [[1.,0.,0.],[.5,.5,0.],[1/3,1/3,1/3]], [[1.,0.],[.5,1.],[4/3,2.]])):
        out, weights = candidate(q, q, values, allowed)
        check_close(weights, torch.tensor(expected_weights, dtype=q.dtype)[None,None], 'hand weights')
        check_close(out, torch.tensor(expected_out, dtype=q.dtype)[None,None], 'hand output')
    return 'Illustrative hand fixture matched (not the missing instructor example)'

In [11]:
pending_or_check('illustrative hand example', lambda: verify_hand_example(scaled_dot_product_attention))

PENDING illustrative hand example: Student attention implementation required


## Multi-head attention — student implementation
Use four biased `nn.Linear(d,d)` attributes named `q_proj`, `k_proj`, `v_proj`, `out_proj`. Constructor: `(d, heads)`, with d divisible by heads. Forward accepts `x` `(B,T,d)` and optional boolean `allowed`, and returns `(output, weights)` with shapes `(B,T,d)` and `(B,heads,T,T)`. Split feature width into heads, call your own attention, then combine heads and apply the output projection. Keep this implementation student-authored. No dropout inside this attention contract; encoder dropout is applied to branch outputs.

In [12]:
def verify_multihead(candidate_class):
    worst = 0.0
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(101)
        for heads in (1, 4, 8):
            d = 16
            custom = candidate_class(d, heads).double()
            reference = nn.MultiheadAttention(d, heads, dropout=0.0, batch_first=True).double()
            with torch.no_grad():
                for index, name in enumerate(('q_proj', 'k_proj', 'v_proj')):
                    projection = getattr(custom, name)
                    projection.weight.copy_(reference.in_proj_weight[index*d:(index+1)*d])
                    projection.bias.copy_(reference.in_proj_bias[index*d:(index+1)*d])
                    assert torch.equal(projection.weight, reference.in_proj_weight[index*d:(index+1)*d])
                    assert torch.equal(projection.bias, reference.in_proj_bias[index*d:(index+1)*d])
                custom.out_proj.load_state_dict(reference.out_proj.state_dict())
                for key, value in custom.out_proj.state_dict().items():
                    assert torch.equal(value, reference.out_proj.state_dict()[key])
                x = torch.randn(2, 5, d, dtype=torch.float64)
                for allowed in (None, torch.ones(5, 5, dtype=torch.bool).tril()):
                    actual, weights = custom(x, allowed)
                    expected, ref_weights = reference(x, x, x, attn_mask=None if allowed is None else ~allowed,
                                                      need_weights=True, average_attn_weights=False)
                    worst = max(worst, check_close(actual, expected, 'multi-head output'),
                                check_close(weights, ref_weights, 'per-head weights'))
            assert sum(p.numel() for p in custom.parameters()) == 4*d*d + 4*d
    return {'max_abs_diff': worst}

In [13]:
class MyMultiHeadAttention(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        # TODO (student): validate dimensions and define four named projections.
        raise NotImplementedError('Student multi-head parameter setup required')

    def forward(self, x, allowed=None):
        # x: (B,T,d); return (B,T,d), (B,heads,T,T).
        # TODO (student): project, split heads, use own attention, combine, project.
        raise NotImplementedError('Student multi-head forward required')

In [14]:
pending_or_check('multi-head', lambda: verify_multihead(MyMultiHeadAttention))

PENDING multi-head: Student multi-head parameter setup required


## Pre-LayerNorm encoder block — student implementation
Constructor `(d, heads, dropout=0.1)`; forward `(x, allowed=None)` returns a tensor `(B,T,d)`.
Use attributes `attention` (your MyMultiHeadAttention), `norm1`, `norm2` (LayerNorm, eps=1e-5), `linear1` (d→4d), `linear2` (4d→d). All projections have biases; LayerNorm has learnable scale and bias. Use ReLU in the feed-forward branch (a documented free choice), with dropout after ReLU and on each residual branch output. Normalize **before** each branch; preserve residual additions. Use your own attention exclusively. Tests turn dropout off for deterministic comparison to a pre-norm PyTorch reference.

Weights: four attention matrices contribute `4d²`; feed-forward matrices contribute `2d*d_ff`. For d=128 and d_ff=512: **196608** weights. Attention biases add 4d; feed-forward biases add d_ff+d; two LayerNorms add 4d. Total: `196608 + 512 + 640 + 512 = 198272`. This count excludes embeddings, positional encodings, final norms and output heads.

In [15]:
def block_parameter_counts(d):
    weights = 4*d*d + 2*d*(4*d)
    return weights, weights + 4*d + (4*d+d) + 4*d

In [16]:
assert block_parameter_counts(128) == (196608, 198272)
reference_count = nn.TransformerEncoderLayer(128, 4, dim_feedforward=512, norm_first=True, batch_first=True)
assert sum(p.numel() for p in reference_count.parameters()) == 198272
print('PASS: block formula and reference total 198272; custom count PENDING')

PASS: block formula and reference total 198272; custom count PENDING


In [17]:
def verify_encoder(candidate_class):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(101)
        d = 128
        custom = candidate_class(d, 4, dropout=0.0).double().eval()
        ref = nn.TransformerEncoderLayer(d, 4, dim_feedforward=4*d, dropout=0.0,
                                        activation='relu', batch_first=True, norm_first=True).double().eval()
        assert sum(p.numel() for p in custom.parameters()) == block_parameter_counts(d)[1]
        with torch.no_grad():
            for name in ('norm1', 'norm2', 'linear1', 'linear2'):
                getattr(custom, name).load_state_dict(getattr(ref, name).state_dict())
            for i, name in enumerate(('q_proj', 'k_proj', 'v_proj')):
                layer = getattr(custom.attention, name)
                layer.weight.copy_(ref.self_attn.in_proj_weight[i*d:(i+1)*d])
                layer.bias.copy_(ref.self_attn.in_proj_bias[i*d:(i+1)*d])
            custom.attention.out_proj.load_state_dict(ref.self_attn.out_proj.state_dict())
        x = torch.randn(2, 5, d, dtype=torch.float64)
        errors = []
        # Grad enabled avoids inference-only fused reference paths; no backward needed.
        for allowed in (None, torch.ones(5, 5, dtype=torch.bool).tril()):
            expected = ref(x, src_mask=None if allowed is None else ~allowed)
            errors.append(check_close(custom(x, allowed), expected, 'pre-LN encoder'))
        return {'params': 198272, 'max_abs_diff': max(errors)}

In [18]:
class MyEncoderBlock(nn.Module):
    def __init__(self, d, heads, dropout=0.1):
        super().__init__()
        # TODO (student): own attention, two norms, 4d feed-forward and dropout.
        raise NotImplementedError('Student encoder parameter setup required')

    def forward(self, x, allowed=None):
        # x and return value: (B,T,d).
        # TODO (student): pre-norm residual attention and feed-forward branches.
        raise NotImplementedError('Student encoder forward required')

In [19]:
pending_or_check('encoder', lambda: verify_encoder(MyEncoderBlock))

PENDING encoder: Student encoder parameter setup required


## Sinusoidal positions — student implementation
Implement `sinusoidal_positions(length, d)` returning float32 `(length,d)`, with d even for this contract. Position p, pair index i: even channel is sin(p/10000^(2i/d)), odd channel is cos(p/10000^(2i/d)). Build this with tensor operations. Add positions to embeddings in later student-authored model code; do not introduce learnable parameters for this variant.

In [20]:
def verify_positions(candidate):
    positions = candidate(7, 8)
    expected = torch.tensor([[math.sin(p / 10000**(2*i/8)) if channel % 2 == 0
                              else math.cos(p / 10000**(2*i/8))
                              for channel in range(8) for i in [channel//2]] for p in range(7)], dtype=torch.float32)
    error = check_close(positions, expected, 'sinusoidal positions')
    assert not positions.requires_grad
    assert not torch.equal(positions[0], positions[1])
    return {'max_abs_diff': error}

In [21]:
def sinusoidal_positions(length, d):
    # Return (length,d), float32, even d, no trainable parameters.
    # TODO (student): implement the sinusoidal formula from the guidance.
    raise NotImplementedError('Student positional encoding required')

In [22]:
pending_or_check('positions', lambda: verify_positions(sinusoidal_positions))

PENDING positions: Student positional encoding required


## Completion and downstream work
`require_problem2_components()` is a strict check: unfinished components raise. Even all these tests passing does not complete Problem 2: student-authored HAR/LM experiments, instructor hand example, ablations, samples and analysis remain required. See `docs/assignment-progress.md` for blockers. No `generated_samples.txt` or experiment rows are produced without actual generation/training.

In [23]:
def require_problem2_components():
    return {'attention': verify_attention(scaled_dot_product_attention),
            'illustrative hand example': verify_hand_example(scaled_dot_product_attention),
            'multi-head': verify_multihead(MyMultiHeadAttention),
            'encoder': verify_encoder(MyEncoderBlock),
            'positions': verify_positions(sinusoidal_positions)}

print('Component status:', P2_RESULTS)
print('Problem 2 incomplete: interfaces and tests only; student code and experiments required.')

Component status: {'attention': 'Blocked', 'illustrative hand example': 'Blocked', 'multi-head': 'Blocked', 'encoder': 'Blocked', 'positions': 'Blocked'}
Problem 2 incomplete: interfaces and tests only; student code and experiments required.
